# Import Librares

In [1]:
import sqlite3
import csv
import pandas as pd

# Define Data Path

In [2]:
data_path = '../MIMIC_Data/physionet.org/files/mimiciii/1.4/1'

# Create a SQL db given MIMIC csv files

In [3]:
conn = sqlite3.connect('mimic_database.db')
file_names = [
    "ADMISSIONS",
    "CALLOUT",
    "CAREGIVERS",
    "CHARTEVENTS",
    "CPTEVENTS",
    "D_CPT",
    "D_ICD_DIAGNOSES",
    "D_ICD_PROCEDURES",
    "D_ITEMS",
    "D_LABITEMS",
    "DATETIMEEVENTS",
    "DIAGNOSES_ICD",
    "DRGCODES",
    "ICUSTAYS",
    "INPUTEVENTS_CV",
    "INPUTEVENTS_MV",
    "LABEVENTS",
    "MICROBIOLOGYEVENTS",
    "NOTEEVENTS",
    "OUTPUTEVENTS",
    "PATIENTS",
    "PRESCRIPTIONS",
    "PROCEDUREEVENTS_MV",
    "PROCEDURES_ICD",
    "SERVICES",
    "TRANSFERS"
]
for file in file_names:
    for chunk in pd.read_csv(f"{data_path}/{file}.csv", chunksize=100000):
        chunk.to_sql(file, conn, index=False, if_exists='append')

conn.close()

C:\Users\patel\AppData\Local\Temp\ipykernel_26028\2926986507.py:31: DtypeWarning: Columns (13) have mixed types. Specify dtype option on import or set low_memory=False.
  for chunk in pd.read_csv(f"{data_path}/{file}.csv", chunksize=100000):
C:\Users\patel\AppData\Local\Temp\ipykernel_26028\2926986507.py:31: DtypeWarning: Columns (5) have mixed types. Specify dtype option on import or set low_memory=False.
  for chunk in pd.read_csv(f"{data_path}/{file}.csv", chunksize=100000):
C:\Users\patel\AppData\Local\Temp\ipykernel_26028\2926986507.py:31: DtypeWarning: Columns (5,7) have mixed types. Specify dtype option on import or set low_memory=False.
  for chunk in pd.read_csv(f"{data_path}/{file}.csv", chunksize=100000):
C:\Users\patel\AppData\Local\Temp\ipykernel_26028\2926986507.py:31: DtypeWarning: Columns (7) have mixed types. Specify dtype option on import or set low_memory=False.
  for chunk in pd.read_csv(f"{data_path}/{file}.csv", chunksize=100000):
C:\Users\patel\AppData\Local\Temp

# Create DB Connection

In [4]:
conn = sqlite3.connect('mimic_database.db')
cur = conn.cursor()

# Create SQL Queries

## Query 1

Patient analysis. Determine the number of male and female patients.

In [14]:
query = '''
            SELECT GENDER, COUNT(*)
            FROM PATIENTS
            GROUP BY GENDER;
        '''
cur.execute(query)
rows = cur.fetchall()
print(rows)

[('F', 20399), ('M', 26121)]


## Query 2

Patient analysis. Determine the age distribution of the patients.

In [19]:
query = '''
            SELECT 
                CASE 
                    WHEN strftime('%Y', DOD) - strftime('%Y', DOB) < 18 THEN '0-17'
                    WHEN strftime('%Y', DOD) - strftime('%Y', DOB) BETWEEN 18 AND 34 THEN '18-34'
                    WHEN strftime('%Y', DOD) - strftime('%Y', DOB) BETWEEN 35 AND 49 THEN '35-49'
                    WHEN strftime('%Y', DOD) - strftime('%Y', DOB) BETWEEN 50 AND 64 THEN '50-64'
                    WHEN strftime('%Y', DOD) - strftime('%Y', DOB) BETWEEN 65 AND 79 THEN '65-79'
                    WHEN DOD IS NOT NULL THEN '80+'
                    ELSE 'Alive'
                END AS age_group,
                COUNT(*) AS patient_count
            FROM PATIENTS
            GROUP BY age_group
            ORDER BY age_group;
        '''
cur.execute(query)
rows = cur.fetchall()
print(rows)

[('0-17', 74), ('18-34', 278), ('35-49', 1010), ('50-64', 3023), ('65-79', 5146), ('80+', 6228), ('Alive', 30761)]


## Query 3

Lab events analysis. Get the lab event's "ITEMID" sorted by the number of times the "FLAG" was "abnormal"

In [20]:
query = '''
            SELECT 
                ITEMID, 
                COUNT(*) AS abnormal_count
            FROM LABEVENTS
            WHERE FLAG = 'abnormal'
            GROUP BY ITEMID
            ORDER BY abnormal_count DESC;
        '''
cur.execute(query)
rows = cur.fetchall()
print(rows)


[(51221, 783689), (51279, 673592), (51222, 667697), (50931, 508914), (51006, 442789), (51277, 346186), (51274, 337209), (50821, 325988), (50912, 321092), (51301, 320247), (51265, 287204), (50893, 268662), (51275, 235026), (51237, 222487), (50970, 219002), (50902, 217537), (50882, 214703), (51248, 208490), (50818, 200752), (50820, 197677), (51249, 157729), (50809, 154439), (50983, 130416), (50804, 129656), (51250, 129401), (51256, 124801), (51244, 115710), (50808, 111158), (50878, 99614), (50971, 95976), (50863, 94076), (50862, 91552), (50861, 89439), (50811, 80401), (50885, 78440), (50813, 77982), (50960, 67117), (50910, 62225), (51003, 59242), (50954, 58100), (50868, 48993), (51493, 38514), (51009, 38333), (50822, 35541), (51516, 23470), (51254, 22654), (50956, 21982), (51200, 21944), (50911, 21896), (51214, 20262), (50883, 19442), (50824, 18770), (51251, 17279), (50867, 17271), (51143, 15623), (51218, 15344), (51257, 14006), (50967, 12688), (50806, 12627), (51144, 12051), (51255, 115

## Query 4

From Query 3, we see that the top lab event that has the highest number of abnormal tests has an ITEMID = 51221

Using the D_LABITEMS table, determine the LABEL associated with ITEMID = 51221

In [23]:
query = '''
            SELECT 
                LABEL
            FROM D_LABITEMS
            WHERE ITEMID = 51221;
        '''
cur.execute(query)
rows = cur.fetchall()
print(rows)

[('Hematocrit',)]


## Query 5

Get the gender distribution for the patients who tested abnormal for lab test with ITEMID = 51221

In [28]:
query = '''
            SELECT 
                p.GENDER, 
                COUNT(*) AS patient_count
            FROM PATIENTS p
            JOIN LABEVENTS l ON p.SUBJECT_ID = l.SUBJECT_ID
            WHERE l.ITEMID = 51221
            AND l.FLAG = 'abnormal'
            GROUP BY p.GENDER;
        '''
cur.execute(query)
rows = cur.fetchall()
print(rows)

[('F', 330697), ('M', 452992)]


# Query 6

Get the count of the drugs that were prescribed to the patients who were abnormal for the lab test "Hematocrit" with ITEMID = 51221

In [24]:
query = '''
            SELECT 
                p.DRUG, 
                COUNT(*) AS prescription_count
            FROM PRESCRIPTIONS p
            JOIN LABEVENTS l ON p.SUBJECT_ID = l.SUBJECT_ID
            WHERE l.ITEMID = 51221
            AND l.FLAG = 'abnormal'
            GROUP BY p.DRUG
            ORDER BY prescription_count DESC;
        '''
cur.execute(query)
rows = cur.fetchall()
print(rows)

[('Insulin', 8008566), ('Potassium Chloride', 7982974), ('D5W', 7523767), ('NS', 6986850), ('Furosemide', 6649484), ('0.9% Sodium Chloride', 5152816), ('Iso-Osmotic Dextrose', 4438734), ('Magnesium Sulfate', 3881055), ('SW', 3389379), ('5% Dextrose', 3305286), ('Metoprolol', 3293109), ('Sodium Chloride 0.9%  Flush', 2859511), ('Acetaminophen', 2751827), ('Lorazepam', 2699593), ('Morphine Sulfate', 2475809), ('Calcium Gluconate', 2242909), ('Vancomycin', 2119770), ('Heparin', 2078598), ('Metoprolol Tartrate', 1979423), ('Warfarin', 1941867), ('HYDROmorphone (Dilaudid)', 1795354), ('Heparin Sodium', 1720842), ('Tacrolimus', 1598630), ('Pantoprazole', 1562473), ('Fentanyl Citrate', 1510102), ('Docusate Sodium', 1465404), ('Bisacodyl', 1430453), ('Vancomycin HCl', 1391501), ('Vial', 1373362), ('LR', 1373106), ('Propofol', 1233434), ('Sodium Bicarbonate', 1122322), ('Senna', 1070843), ('Aspirin', 1010024), ('Levofloxacin', 965010), ('Ondansetron', 930058), ('Bag', 929393), ('Albuterol 0.083

# Query 7

Look at the religion distrbution for patients who were abnormal for the lab test "Hematocrit" with ITEMID = 51221

## Query 8

Look at the insurance distribution for patients who were abnormal for the lab test "Hematocrit" with ITEMID = 51221

# Close DB Connection

In [ ]:
conn.close()